# ragk8s — reference generator `qwen2.5:7b-instruct` (Ollama) on Kaggle

Generates the answers of the reference model on the stored contexts of the best
configuration, with exactly the same prompt and decoding options as local runs.

**Before running**
1. *Settings → Accelerator*: **GPU T4 x2**. *Settings → Internet*: **on** (phone-verified account).
2. *Add Input*: the dataset made from `kaggle/export/contexts-fixed_hybrid_rerank.zip`
   (created locally with `python -m ragk8s kaggle export --preset best`).
3. *Run All*. About 10–20 minutes, mostly the model download.

**After running**: download `ragk8s-output.zip` from the *Output* panel, then locally:
`python -m ragk8s kaggle import <path-to>/ragk8s-output.zip`.

In [ ]:
import json, pathlib, subprocess, sys, time

manifests = list(pathlib.Path("/kaggle/input").rglob("manifest.json"))
assert len(manifests) == 1, f"attach exactly one ragk8s export dataset, found {manifests}"
DATA = manifests[0].parent
MANIFEST = json.loads(manifests[0].read_text())
print(json.dumps(MANIFEST, indent=2))
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], check=True)

Install `ragk8s` at the exact commit recorded in the export (no heavy dependencies are needed to generate).

In [ ]:
pkg = f"git+{MANIFEST['repo']}@{MANIFEST['commit']}"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", pkg], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ollama==0.6.3"], check=True)
import ragk8s
print("ragk8s", ragk8s.__version__)

Install and start Ollama, then pull the reference model.

In [ ]:
import os, urllib.request
subprocess.run("apt-get -qq update && apt-get -qq install -y zstd > /dev/null", shell=True, check=True)
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
server = subprocess.Popen(["ollama", "serve"], stdout=open("/kaggle/working/ollama.log", "w"),
                          stderr=subprocess.STDOUT, env={**os.environ, "OLLAMA_KEEP_ALIVE": "-1"})
for _ in range(60):
    try:
        print(urllib.request.urlopen("http://127.0.0.1:11434/api/version").read().decode())
        break
    except OSError:
        time.sleep(1)
else:
    raise RuntimeError("Ollama did not start, see /kaggle/working/ollama.log")
subprocess.run(["ollama", "pull", "qwen2.5:7b-instruct"], check=True)
subprocess.run(["ollama", "list"], check=True)

Copy the stored contexts and run the generation. The answer cache lives in
`/kaggle/working/rag/.cache`, so re-running this cell after an interruption resumes.

In [ ]:
import shutil
WORK = pathlib.Path("/kaggle/working/rag")
for sub in ("results/retrieval", "eval"):
    shutil.copytree(DATA / sub, WORK / sub, dirs_exist_ok=True)
os.chdir(WORK)
subprocess.run([sys.executable, "-m", "ragk8s", "generate", "--provider", "ollama", "--preset", "best"], check=True)

In [ ]:
meta = json.loads(next(WORK.glob("results/runs/*/run_*.meta.json")).read_text())
print({k: meta[k] for k in ("config", "n_questions", "model", "model_digest", "ollama_version",
                            "tokens_mean", "generation_seconds_mean", "platform", "created_at")})

Package the outputs for download (runs + answer cache).

In [ ]:
import zipfile
out = pathlib.Path("/kaggle/working/ragk8s-output.zip")
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in ("results/runs", ".cache/generations"):
        for f in (WORK / folder).rglob("*"):
            if f.is_file():
                z.write(f, f.relative_to(WORK))
print(out, round(out.stat().st_size / 1024), "KiB")